In [43]:
import os
print("je m'execute depuis : ", os.getcwd())
print("Le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data : ", os.listdir("../data"))

je m'execute depuis :  c:\Users\kelly\Desktop\WCS\Projets\json-api-geocodage\notebooks
Le dossier data existe : True
le fichier existe : True
contenu de ../data :  ['restaurants.json', 'villes.json']


In [44]:
import json

with open ("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [45]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [46]:
for r in data["restaurants"] : 
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r["nom"]:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [47]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [48]:
import pandas as pd
df_test = pd.json_normalize(data)
print(df_test.shape)
print(df_test.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [49]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [50]:
import requests
import time
headers = {"User-Agent" : "bootcamp-simplon-data/1.0"}
url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params = {"q": "20 avenue de Ségur 75007 Paris", "limit": 1}, headers = headers)
time.sleep(1)
print(r.status_code)

200


In [51]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [52]:
print(r.url)

https://api-adresse.data.gouv.fr/search/?q=20+avenue+de+S%C3%A9gur+75007+Paris&limit=1


In [53]:
headers = {"User-Agent" : "bootcamp-simplon-data/1.0"}
url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params = {"q": "Catane"}, headers = headers)
time.sleep(1)
print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


In [54]:
def interroger(url, params, headers=None) : 
    try : 
        requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur : ", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [1.197596, 43.196477]}, 'properties': {'label': 'Catane 31310 Montesquieu-Volvestre', 'score': 0.9437199999999999, 'type': 'locality', 'importance': 0.38092, 'id': '31375_dpvo40', 'banId': 'e8b353b0-ed02-40ec-9082-f01990905a83', 'name': 'Catane', 'postcode': '31310', 'citycode': '31375', 'x': 553410.29, 'y': 6234701.67, 'city': 'Montesquieu-Volvestre', 'context': '31, Haute-Garonne, Occitanie', 'depcode': '31', 'locality': 'Catane', '_type': 'address'}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [5.701625, 45.180037]}, 'properties': {'label': 'Pont de Catane 38000 Grenoble', 'score': 0.6888772727272726, 'type': 'locality', 'importance': 0.57765, 'id': '38185_1323', 'banId': 'fad65751-31b8-44ce-bca9-0078d086201a', 'name': 'Pont de Catane', 'postcode': '38000', 'citycode': '38185', 'x': 912160.84, 'y': 6457044.1, 'city': 'Grenoble', 'context': '38, Isère

In [57]:
url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q" : "20 avenue de Ségur 75007 Paris", "limit" : 1})
reponse = r.json()

print(json.dumps(reponse, ensure_ascii=False, indent=2)[:600])

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.308628,
          48.850699
        ]
      },
      "properties": {
        "label": "20 Avenue de Ségur 75007 Paris",
        "score": 0.9716281818181818,
        "housenumber": "20",
        "id": "75107_8909_00020",
        "banId": "c575af14-f697-464c-b70b-914cedb239a4",
        "name": "20 Avenue de Ségur",
        "postcode": "75007",
        "citycode": "75107",
        "x": 649266.35,
        "y": 6861406.23,
        "city": 


In [59]:
f = reponse["features"][0]
lon, lat = f["geometry"]["coordinates"]
print("longitude :", lon, "latitude :", lat)
print("label     :", f["properties"]["label"])

longitude : 2.308628 latitude : 48.850699
label     : 20 Avenue de Ségur 75007 Paris


In [61]:
df=pd.json_normalize(reponse, record_path="features")
print(df.columns.tolist())

['type', 'geometry.type', 'geometry.coordinates', 'properties.label', 'properties.score', 'properties.housenumber', 'properties.id', 'properties.banId', 'properties.name', 'properties.postcode', 'properties.citycode', 'properties.x', 'properties.y', 'properties.city', 'properties.district', 'properties.context', 'properties.type', 'properties.importance', 'properties.depcode', 'properties.street', 'properties._type']


In [62]:
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
print(df[["properties.label", "lat", "lon"]])

                 properties.label        lat       lon
0  20 Avenue de Ségur 75007 Paris  48.850699  2.308628


In [63]:
def geocoder_fr(adresse): 
    """Renvoie (latitude, longitude) pour une adresse française, ou (None, None)."""
    url = "https://api-adresse.data.gouv.fr/search/"
    try : 
     r=requests.get(url, params={"q" : adresse, "limit" : 1}, timeout=10)
     r.raise_for_status()
     features=r.json()["features"]
     if not features : 
        return None, None
     lon, lat = features[0]["geometry"]["coordinates"]
     return lat, lon
    except requests.exceptions.RequestException:
       return None,None

In [64]:
print(geocoder_fr("20 avenue de Ségur 75007 Paris"))
print(geocoder_fr("zzzzz qwerty 99999"))

(48.850699, 2.308628)
(None, None)


In [69]:
url = "https://nominatim.openstreetmap.org/search"
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (https://github.com/kelly-lachance)"}
params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}
r = requests.get(url, params=params, headers=headers, timeout=10)
resultat = r.json()

print(float(resultat[0]["lat"]), float(resultat[0]["lon"]))
print(resultat[0]["display_name"])

37.5111836 15.094914
Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia


In [68]:
print(json.dumps(resultat, ensure_ascii=False, indent=2)[:600])

[
  {
    "place_id": 59502264,
    "licence": "Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright",
    "osm_type": "way",
    "osm_id": 409895581,
    "lat": "37.5111836",
    "lon": "15.0949140",
    "class": "highway",
    "type": "residential",
    "place_rank": 26,
    "importance": 0.05340454642557409,
    "addresstype": "residential",
    "name": "Via Pietro Mascagni",
    "display_name": "Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia",
    "boundingbox": [
      "37.5111306",
      "37.5112366",
      "15.0948817",
     


In [70]:
df=pd.json_normalize(resultat)
print(df[["lat","lon","display_name"]])

          lat         lon                                       display_name
0  37.5111836  15.0949140  Via Pietro Mascagni, Picanello-Ognina-Barriera...


In [71]:
def geocoder_monde(adresse):
    url="https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (https://github.com/kelly-lachance)"}
    params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}
    try : 
        r=requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats=r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException : 
        return None, None
    finally : 
        time.sleep(1)

In [72]:
debut = time.time()
print(geocoder_monde("54 Via Pietro Mascagni, Catania, Italy"))
print(geocoder_monde("Piazza del Duomo, Catania, Italy"))
print(f"durée totale : {time.time() - debut:.1f} s")


(37.5111836, 15.094914)
(37.5111836, 15.094914)
durée totale : 2.6 s


In [74]:
import folium
point = [37.5112, 15.0949]
m=folium.Map(location=point, zoom_start=14)
folium.Marker(location=point, popup="Un bon restaurant").add_to(m)
m